# Module 2: Data Exploration and Data Preprocessing
### Personalized TED Talks Recommendation System Using Machine Learning
**Infosys Springboard Virtual Internship 7.0**

---

## 1. Introduction and Objectives
In **Module 2**, we explore the TED Talks datasets, evaluate data quality, generate rich visualizations, and clean the data for downstream feature engineering and recommendation models.

**Key Objectives:**
1. **Initial Data Assessment:** Audit shapes, data types, descriptive statistics, missing values, duplicates, and blank/padded strings.
2. **Data Quality Documentation:** Export an automated Data Quality Report (`reports/data_quality_report.csv`).
3. **Exploratory Data Analysis (EDA):** Generate and save 11 publication-grade visualizations using Matplotlib and Seaborn.
4. **Data Cleaning & Wrangling:**
   - Preserve raw immutable datasets.
   - Strip leading/trailing whitespace and trailing `\n` from URLs and text fields.
   - Impute missing `speaker_occupation` (fill with `'Unknown'`).
   - Deduplicate `transcripts.csv` (remove 3 exact duplicate rows).
   - Safely parse string-encoded Python lists (`tags`) and lists of dicts (`ratings`).
   - Clean transcript text (remove stage directions like `(Laughter)` / `(Applause)`, lowercase, normalize whitespace).
   - Engineer useful features (`clean_transcript`, `word_count`, `duration_minutes`, `film_year`, `published_year`, `top_rating`).
5. **Before-and-After Validation:** Verify shape deltas, missing value resolutions, and record retention.
6. **Persist Cleaned Datasets:** Save `data/ted_main_cleaned.csv` and `data/transcripts_cleaned.csv`.


In [1]:
import os
import sys
import ast
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set publication-quality plot styling
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 16,
    "figure.dpi": 150,
})

PRIMARY_COLOR = "#E62B1E"    # TED Red
SECONDARY_COLOR = "#1D3557"  # Deep Navy
ACCENT_COLOR = "#457B9D"     # Slate Blue

# Configure paths and sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

data_dir = project_root / "data"
reports_dir = project_root / "reports"
eda_dir = reports_dir / "eda"
eda_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root.resolve()}")
print(f"Data dir: {data_dir.resolve()}")
print("Environment setup completed successfully.")


Project root: E:\infosys\TED Talks Recommendation System
Data dir: E:\infosys\TED Talks Recommendation System\data
Environment setup completed successfully.


## 2. Ingestion of Raw Datasets

In [2]:
df_main_raw = pd.read_csv(data_dir / "ted_main.csv")
df_trans_raw = pd.read_csv(data_dir / "transcripts.csv")

print(f"Raw ted_main shape: {df_main_raw.shape}")
print(f"Raw transcripts shape: {df_trans_raw.shape}")


Raw ted_main shape: (2550, 17)
Raw transcripts shape: (2467, 2)


## 3. Initial Data Assessment & Data Quality Audit
Let us evaluate missing values, data types, descriptive statistics, and whitespace issues across both datasets.

In [3]:
print("=== Missing Values in ted_main.csv ===")
main_nulls = df_main_raw.isnull().sum()
main_nulls_pct = (df_main_raw.isnull().sum() / len(df_main_raw)) * 100
main_missing_df = pd.DataFrame({"Missing_Count": main_nulls, "Missing_Pct (%)": main_nulls_pct.round(3)})
display(main_missing_df[main_missing_df["Missing_Count"] > 0])

print("\n=== Missing Values in transcripts.csv ===")
trans_nulls = df_trans_raw.isnull().sum()
trans_nulls_pct = (df_trans_raw.isnull().sum() / len(df_trans_raw)) * 100
trans_missing_df = pd.DataFrame({"Missing_Count": trans_nulls, "Missing_Pct (%)": trans_nulls_pct.round(3)})
display(trans_missing_df)


=== Missing Values in ted_main.csv ===


,Missing_Count,Missing_Pct (%)
speaker_occupation,6,0.235



=== Missing Values in transcripts.csv ===


,Missing_Count,Missing_Pct (%)
transcript,0,0.0
url,0,0.0


In [4]:
print("=== Numerical Descriptive Statistics (ted_main.csv) ===")
df_main_raw.describe().round(2)


=== Numerical Descriptive Statistics (ted_main.csv) ===


,comments,duration,film_date,languages,num_speaker,published_date,views
count,2550.00,2550.00,2.550000e+03,2550.00,2550.00,2.550000e+03,2550.00
mean,191.56,826.51,1.321928e+09,27.33,1.03,1.343525e+09,1698297.48
std,282.32,374.01,1.197391e+08,9.56,0.21,9.464009e+07,2498479.37
min,2.00,135.00,7.464960e+07,0.00,1.00,1.151367e+09,50443.00
25%,63.00,577.00,1.257466e+09,23.00,1.00,1.268463e+09,755792.75
50%,118.00,848.00,1.333238e+09,28.00,1.00,1.340935e+09,1124523.50
75%,221.75,1046.75,1.412964e+09,33.00,1.00,1.423432e+09,1700759.50
max,6404.00,5256.00,1.503792e+09,72.00,5.00,1.506092e+09,47227110.00


In [5]:
print("=== Whitespace / Newline Padding Audit ===")
padding_report = []
for col in df_main_raw.select_dtypes(include="object").columns:
    s = df_main_raw[col].dropna().astype(str)
    blank_cnt = (s.str.strip() == "").sum()
    padded_cnt = (s != s.str.strip()).sum()
    padding_report.append({"Dataset": "ted_main", "Column": col, "Blank_Strings": blank_cnt, "Padded_Records": padded_cnt})

for col in df_trans_raw.select_dtypes(include="object").columns:
    s = df_trans_raw[col].dropna().astype(str)
    blank_cnt = (s.str.strip() == "").sum()
    padded_cnt = (s != s.str.strip()).sum()
    padding_report.append({"Dataset": "transcripts", "Column": col, "Blank_Strings": blank_cnt, "Padded_Records": padded_cnt})

display(pd.DataFrame(padding_report))


=== Whitespace / Newline Padding Audit ===


,Dataset,Column,Blank_Strings,Padded_Records
0,ted_main,description,0,53
1,ted_main,event,0,1
2,ted_main,main_speaker,0,6
3,ted_main,name,0,1
4,ted_main,ratings,0,0
5,ted_main,related_talks,0,0
6,ted_main,speaker_occupation,0,42
7,ted_main,tags,0,0
8,ted_main,title,0,8
9,ted_main,url,0,2550


### 3.1 Generate Automated Data Quality Report
We compile a structured audit report summarizing every column's type, non-null count, missing count, missing percentage, blank strings, whitespace padding, unique values, and diagnostic notes.

In [6]:
# Import data quality report function from src or compute directly
from src.data_processing import generate_data_quality_report

quality_report_csv = reports_dir / "data_quality_report.csv"
quality_df = generate_data_quality_report(df_main_raw, df_trans_raw, quality_report_csv)
print(f"Data quality report saved to: {quality_report_csv.resolve()}")
quality_df


Data quality report saved to: E:\infosys\TED Talks Recommendation System\reports\data_quality_report.csv


,Dataset,Column,Data_Type,Total_Records,Non_Null_Count,Missing_Count,Missing_Pct,Blank_Strings,Whitespace_Padded,Unique_Values,Sample_Value,Quality_Assessment
0,ted_main.csv,comments,int64,2550,2550,0,0.000,0,0,559,4553,Clean
1,ted_main.csv,description,object,2550,2550,0,0.000,0,53,2550,Sir Ken Robinson makes an entertainin...,53 whitespace/newline padded
2,ted_main.csv,duration,int64,2550,2550,0,0.000,0,0,1083,1164,Clean
3,ted_main.csv,event,object,2550,2550,0,0.000,0,1,355,TED2006,1 whitespace/newline padded
4,ted_main.csv,film_date,int64,2550,2550,0,0.000,0,0,735,1140825600,Unix epoch timestamps in seconds
5,ted_main.csv,languages,int64,2550,2550,0,0.000,0,0,66,60,Clean
6,ted_main.csv,main_speaker,object,2550,2550,0,0.000,0,6,2156,Ken Robinson,6 whitespace/newline padded
7,ted_main.csv,name,object,2550,2550,0,0.000,0,1,2550,Ken Robinson: Do schools kill creativ...,1 whitespace/newline padded
8,ted_main.csv,num_speaker,int64,2550,2550,0,0.000,0,0,5,1,Clean
9,ted_main.csv,published_date,int64,2550,2550,0,0.000,0,0,2490,1151367060,Unix epoch timestamps in seconds


## 4. Exploratory Data Analysis (EDA)
We now generate and analyze 11 comprehensive visualizations representing the distributions, correlations, and characteristics of TED Talks.

### 4.1 Most Frequently Occurring TED Talk Tags
TED talks are categorized by subject tags. Let us extract and rank the top 20 most frequent tags.

In [7]:
tag_counts = Counter()
for t_str in df_main_raw["tags"]:
    try:
        t_list = ast.literal_eval(t_str)
        for t in t_list:
            tag_counts[str(t).strip().lower()] += 1
    except Exception:
        pass

top_tags = tag_counts.most_common(20)
tags_df = pd.DataFrame(top_tags, columns=["Tag", "Frequency"])

plt.figure(figsize=(10, 6))
bars = plt.barh(tags_df["Tag"][::-1], tags_df["Frequency"][::-1], color=PRIMARY_COLOR, alpha=0.9)
plt.title("Top 20 Most Frequent TED Talk Tags", fontsize=14, weight="bold", pad=12)
plt.xlabel("Frequency (Number of Talks)")
plt.ylabel("Tag Name")
plt.xlim(0, max(tags_df["Frequency"]) * 1.12)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 12, bar.get_y() + bar.get_height() / 2, f"{int(w):,}", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.savefig(eda_dir / "01_top_tags.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\1833333068.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.2 Distribution of Talk Durations
The core TED format historically limits talks to 18 minutes (1,080 seconds). Let us inspect the duration distribution in minutes.

In [8]:
durations_min = df_main_raw["duration"] / 60.0
median_dur = durations_min.median()
mean_dur = durations_min.mean()

plt.figure(figsize=(10, 5.5))
sns.histplot(durations_min, kde=True, bins=40, color=SECONDARY_COLOR, edgecolor="white", alpha=0.85)
plt.axvline(18.0, color=PRIMARY_COLOR, linestyle="--", linewidth=2, label="TED 18-min Target")
plt.axvline(median_dur, color="#2A9D8F", linestyle="-.", linewidth=2, label=f"Median ({median_dur:.1f} min)")
plt.axvline(mean_dur, color="#E76F51", linestyle=":", linewidth=2, label=f"Mean ({mean_dur:.1f} min)")

plt.title("Distribution of TED Talk Durations (Minutes)", fontsize=14, weight="bold", pad=12)
plt.xlabel("Duration (Minutes)")
plt.ylabel("Number of Talks")
plt.xlim(0, 45)
plt.legend(frameon=True, facecolor="white")

plt.tight_layout()
plt.savefig(eda_dir / "02_duration_distribution.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\2888379411.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.3 Distribution of Talk Views
TED talk view counts span from ~50,000 to over 47 million. We examine both linear (in millions) and logarithmic scales.

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

views_m = df_main_raw["views"] / 1e6
sns.histplot(views_m, bins=45, color=ACCENT_COLOR, kde=True, ax=ax1, edgecolor="white")
ax1.set_title("Distribution of Views (Linear Scale in Millions)", weight="bold")
ax1.set_xlabel("Views (Millions)")
ax1.set_ylabel("Talk Count")
ax1.axvline(views_m.median(), color=PRIMARY_COLOR, linestyle="--", label=f"Median: {views_m.median():.2f}M")
ax1.legend()

sns.histplot(df_main_raw["views"], bins=45, log_scale=True, color=SECONDARY_COLOR, kde=True, ax=ax2, edgecolor="white")
ax2.set_title("Distribution of Views (Logarithmic Scale)", weight="bold")
ax2.set_xlabel("Views (Log Scale)")
ax2.set_ylabel("Talk Count")

plt.tight_layout()
plt.savefig(eda_dir / "03_views_distribution.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\2006347634.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.4 Aggregate Distribution of Ratings
Audience members rate TED talks across 14 emotional dimensions (e.g. *Inspiring*, *Informative*, *Fascinating*).

In [10]:
rating_totals = Counter()
for r_str in df_main_raw["ratings"]:
    try:
        r_list = ast.literal_eval(r_str)
        for item in r_list:
            if isinstance(item, dict):
                rating_totals[item.get("name", "Unknown")] += item.get("count", 0)
    except Exception:
        pass

ratings_df = pd.DataFrame(rating_totals.most_common(), columns=["Rating", "Total_Votes"])

plt.figure(figsize=(10, 6))
bars = plt.barh(ratings_df["Rating"][::-1], ratings_df["Total_Votes"][::-1] / 1000, color="#2A9D8F", alpha=0.9)
plt.title("Aggregate TED Talk Audience Ratings (Thousands of Votes)", fontsize=14, weight="bold", pad=12)
plt.xlabel("Total Votes (in Thousands)")
plt.ylabel("Rating Category")
plt.xlim(0, (max(ratings_df["Total_Votes"]) / 1000) * 1.15)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 15, bar.get_y() + bar.get_height() / 2, f"{int(w):,}k", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.savefig(eda_dir / "04_ratings_distribution.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\1138435075.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.5 Data Completeness & Missing Values Audit

In [11]:
missing_dict = {
    "speaker_occupation (raw)": 6,
    "speaker_occupation (cleaned)": 0,
    "transcripts (unmatched)": 86,
    "tags (missing)": 0,
    "views (missing)": 0,
    "duration (missing)": 0,
    "title (missing)": 0,
    "description (missing)": 0
}
missing_df = pd.DataFrame(list(missing_dict.items()), columns=["Field", "Missing_Count"])

plt.figure(figsize=(10, 5))
bar_colors = ["#E76F51" if c > 0 else "#2A9D8F" for c in missing_df["Missing_Count"]]
bars = plt.bar(missing_df["Field"], missing_df["Missing_Count"], color=bar_colors, width=0.6)
plt.title("Data Completeness and Missing Values Audit", fontsize=14, weight="bold", pad=12)
plt.xlabel("Field Evaluated")
plt.ylabel("Missing Count (Rows)")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 100)

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 2, f"{int(h)}", ha="center", va="bottom", fontsize=10, weight="bold")

plt.tight_layout()
plt.savefig(eda_dir / "05_missing_values.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\1899356982.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.6 Transcript Word Count Distribution
We examine the word count distribution across all transcripts.

In [12]:
word_counts = df_trans_raw["transcript"].fillna("").astype(str).apply(lambda x: len(x.split()))
median_words = word_counts.median()
mean_words = word_counts.mean()

plt.figure(figsize=(10, 5.5))
sns.histplot(word_counts, kde=True, bins=40, color="#264653", edgecolor="white", alpha=0.85)
plt.axvline(median_words, color=PRIMARY_COLOR, linestyle="--", linewidth=2, label=f"Median ({int(median_words):,} words)")
plt.axvline(mean_words, color="#E76F51", linestyle="-.", linewidth=2, label=f"Mean ({int(mean_words):,} words)")

plt.title("Distribution of Transcript Word Counts", fontsize=14, weight="bold", pad=12)
plt.xlabel("Word Count per Talk")
plt.ylabel("Number of Talks")
plt.legend(frameon=True, facecolor="white")

plt.tight_layout()
plt.savefig(eda_dir / "06_transcript_length_distribution.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\4254565629.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.7 Top 15 Most Frequent TED Talk Speakers

In [13]:
top_speakers = df_main_raw["main_speaker"].str.strip().value_counts().head(15).reset_index()
top_speakers.columns = ["Speaker", "Talk_Count"]

plt.figure(figsize=(10, 5.5))
bars = plt.barh(top_speakers["Speaker"][::-1], top_speakers["Talk_Count"][::-1], color=SECONDARY_COLOR, alpha=0.9)
plt.title("Top 15 Most Frequent TED Talk Speakers", fontsize=14, weight="bold", pad=12)
plt.xlabel("Number of Talks")
plt.ylabel("Speaker Name")
plt.xlim(0, max(top_speakers["Talk_Count"]) + 1.5)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.15, bar.get_y() + bar.get_height() / 2, f"{int(w)}", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.savefig(eda_dir / "07_top_speakers.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\1757950937.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.8 Top 15 Speaker Occupations

In [14]:
top_occ = df_main_raw["speaker_occupation"].dropna().str.strip().value_counts().head(15).reset_index()
top_occ.columns = ["Occupation", "Count"]

plt.figure(figsize=(10, 5.5))
bars = plt.barh(top_occ["Occupation"][::-1], top_occ["Count"][::-1], color=ACCENT_COLOR, alpha=0.9)
plt.title("Top 15 TED Speaker Occupations", fontsize=14, weight="bold", pad=12)
plt.xlabel("Number of Speakers")
plt.ylabel("Occupation")
plt.xlim(0, max(top_occ["Count"]) + 6)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.8, bar.get_y() + bar.get_height() / 2, f"{int(w)}", va="center", ha="left", fontsize=9)

plt.tight_layout()
plt.savefig(eda_dir / "08_top_occupations.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\3420383831.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.9 Views vs. Comments Correlation

In [15]:
plt.figure(figsize=(9, 6))
corr = df_main_raw[["views", "comments"]].corr().iloc[0, 1]

sns.regplot(
    x="views", y="comments", data=df_main_raw,
    scatter_kws={"alpha": 0.45, "color": SECONDARY_COLOR, "s": 25},
    line_kws={"color": PRIMARY_COLOR, "linewidth": 2}
)
plt.xscale("log")
plt.yscale("log")
plt.title(f"Views vs. Comments Correlation (Pearson r = {corr:.2f})", fontsize=14, weight="bold", pad=12)
plt.xlabel("Views (Log Scale)")
plt.ylabel("Comments (Log Scale)")

plt.tight_layout()
plt.savefig(eda_dir / "09_views_vs_comments.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\1960833232.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.10 Annual Publication Trend (2006 - 2017)

In [16]:
pub_years = pd.to_datetime(df_main_raw["published_date"], unit="s").dt.year.value_counts().sort_index().reset_index()
pub_years.columns = ["Year", "Talk_Count"]

plt.figure(figsize=(10, 5))
plt.plot(pub_years["Year"], pub_years["Talk_Count"], marker="o", linewidth=2.5, color=PRIMARY_COLOR, markersize=7)
plt.fill_between(pub_years["Year"], pub_years["Talk_Count"], color=PRIMARY_COLOR, alpha=0.15)
plt.title("TED Talks Published by Year (2006 - 2017)", fontsize=14, weight="bold", pad=12)
plt.xlabel("Year Published")
plt.ylabel("Number of Talks")
plt.xticks(pub_years["Year"])

for _, row in pub_years.iterrows():
    plt.text(row["Year"], row["Talk_Count"] + 7, f"{int(row['Talk_Count'])}", ha="center", fontsize=9, weight="bold")

plt.ylim(0, max(pub_years["Talk_Count"]) * 1.18)
plt.tight_layout()
plt.savefig(eda_dir / "10_talks_by_year.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\623886718.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4.11 Transcript Coverage Analysis (Donut Chart)

In [17]:
clean_main_urls = set(df_main_raw["url"].str.strip())
clean_trans_urls = set(df_trans_raw["url"].str.strip())
common = len(clean_main_urls.intersection(clean_trans_urls))
unmatched = len(clean_main_urls - clean_trans_urls)

labels = [
    f"Spoken Talks with Transcripts\n({common:,} talks - {common/len(clean_main_urls):.1%})",
    f"Music / Dance / Art Demonstrations\n({unmatched:,} talks - {unmatched/len(clean_main_urls):.1%})"
]
sizes = [common, unmatched]
colors = ["#2A9D8F", "#E76F51"]

fig, ax = plt.subplots(figsize=(7, 6))
wedges, texts = ax.pie(
    sizes, labels=labels, colors=colors, startangle=140,
    wedgeprops=dict(width=0.4, edgecolor="white", linewidth=2),
    textprops=dict(fontsize=10, weight="bold")
)
ax.set_title("Transcript Coverage in TED Dataset", fontsize=14, weight="bold", pad=14)

plt.tight_layout()
plt.savefig(eda_dir / "11_transcript_coverage.png", bbox_inches="tight")
plt.show()


C:\Users\jarap\AppData\Local\Temp\ipykernel_25836\2906751401.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Data Cleaning & Preprocessing Pipeline
We now apply our production-grade cleaning transformations via functions defined in `src/data_processing.py`:
1. Stripping trailing `\n` and whitespace from URLs and text fields.
2. Imputing missing `speaker_occupation` with `'Unknown'`.
3. Deduplicating `transcripts.csv` by removing the 3 exact duplicate rows.
4. Parsing tags into clean token lists and strings.
5. Extracting `total_ratings` and `top_rating` from JSON-like rating dicts.
6. Cleaning transcript text (removing stage directions, lowercase, normalizing whitespace).
7. Flagging `has_transcript`.

In [18]:
from src.data_processing import (
    preprocess_ted_main,
    preprocess_transcripts,
    validate_cleaning
)

# Common clean URLs set
common_urls = set(df_trans_raw["url"].str.strip())

# Clean DataFrames
df_main_clean = preprocess_ted_main(df_main_raw, common_urls=common_urls)
df_trans_clean = preprocess_transcripts(df_trans_raw)

print(f"Cleaned ted_main shape: {df_main_clean.shape}")
print(f"Cleaned transcripts shape: {df_trans_clean.shape}")


Cleaned ted_main shape: (2550, 28)
Cleaned transcripts shape: (2464, 7)


### 5.1 Preview Cleaned Metadata (`ted_main_cleaned.csv`)

In [19]:
df_main_clean[["title", "main_speaker", "speaker_occupation", "clean_tags", "top_rating", "duration_minutes", "has_transcript"]].head()


,title,main_speaker,speaker_occupation,clean_tags,top_rating,duration_minutes,has_transcript
0,Do schools kill creativity?,Ken Robinson,Author/educator,children creativity culture dance education pa...,Inspiring,19.40,1
1,Averting the climate crisis,Al Gore,Climate advocate,alternative energy cars climate change culture...,Funny,16.28,1
2,Simplicity sells,David Pogue,Technology columnist,computers entertainment interface design media...,Funny,21.43,1
3,Greening the ghetto,Majora Carter,Activist for environmental justice,macarthur grant activism business cities envir...,Inspiring,18.60,1
4,The best stats you've ever seen,Hans Rosling,Global health expert; data visionary,africa asia google demo economics global devel...,Informative,19.83,1


### 5.2 Preview Cleaned Transcripts (`transcripts_cleaned.csv`)

In [20]:
df_trans_clean[["url", "word_count", "clean_word_count", "clean_transcript"]].head()


,url,word_count,clean_word_count,clean_transcript
0,https://www.ted.com/talks/ken_robinson_says_sc...,3066,3120,good morning how are you it's been great hasn'...
1,https://www.ted.com/talks/al_gore_on_averting_...,2089,2123,thank you so much chris and it's truly a great...
2,https://www.ted.com/talks/david_pogue_says_sim...,3253,3288,hello voice mail my old friend i've called for...
3,https://www.ted.com/talks/majora_carter_s_tale...,3015,3095,if you're here today and i'm very happy that y...
4,https://www.ted.com/talks/hans_rosling_shows_t...,3121,3164,about 10 years ago i took on the task to teach...


## 6. Before-and-After Validation
We formally validate the before-and-after states to verify data integrity.

In [21]:
val_csv_path = reports_dir / "before_after_validation.csv"
val = validate_cleaning(df_main_raw, df_main_clean, df_trans_raw, df_trans_clean, output_csv_path=val_csv_path)
print(f"Before-and-after validation report saved to: {val_csv_path.resolve()}")
display(val["table"])

print("\n=== Data Cleaning Decisions & Rationale ===")
display(pd.DataFrame(val["cleaning_decisions"]))


Before-and-after validation report saved to: E:\infosys\TED Talks Recommendation System\reports\before_after_validation.csv


,Dataset,Metric,Raw_Value,Cleaned_Value,Delta,Rationale_and_Notes
0,ted_main.csv,Total Records (Rows),2550,2550,0,100% of metadata records preserved (0 deleted)
1,ted_main.csv,Total Features (Columns),17,28,11,"Engineered clean_tags, tag_count, total_rating..."
2,ted_main.csv,Missing Values Count,6,0,-6,Imputed 6 missing speaker_occupation rows with...
3,ted_main.csv,Duplicate Rows,0,0,0,Zero duplicate records in metadata
4,ted_main.csv,Whitespace-Padded URLs,2550,0,-2550,Stripped trailing newline (\n) and whitespace ...
5,transcripts.csv,Total Records (Rows),2467,2464,-3,"Removed 3 confirmed exact duplicate rows; 2,46..."
6,transcripts.csv,Total Features (Columns),2,7,5,"Engineered clean_transcript, char_count, word_..."
7,transcripts.csv,Missing Values Count (Transcript),0,0,0,Zero missing verbatim transcripts
8,transcripts.csv,Duplicate Rows,3,0,-3,Exact duplicate transcript rows dropped (1 uni...
9,transcripts.csv,Whitespace-Padded URLs,2467,0,-2467,Stripped trailing newline (\n) and whitespace ...



=== Data Cleaning Decisions & Rationale ===


,Item,Action,Rationale
0,Metadata URLs & Text,Stripped trailing newline (\n) and whitespace ...,Ensures exact string matching between metadata...
1,Missing speaker_occupation,Imputed 6 missing records with 'Unknown',Preserves talk metadata without inventing spec...
2,Duplicate Transcripts,Removed 3 exact duplicate URL records in trans...,Eliminates redundant identical transcripts; re...
3,Tags Parsing,Parsed string lists safely with ast.literal_ev...,Enables accurate tag frequency analysis and do...
4,Transcript Text Cleaning,Generated clean_transcript with stage notes (L...,Removes non-lexical acoustic noise while prese...
5,Date & Rating Feature Engineering,"Converted Unix timestamps to datetimes, extrac...",Enables temporal analysis and categorical base...


## 7. Persisting Cleaned Datasets

In [22]:
clean_main_csv = data_dir / "ted_main_cleaned.csv"
clean_trans_csv = data_dir / "transcripts_cleaned.csv"

df_main_clean.to_csv(clean_main_csv, index=False)
df_trans_clean.to_csv(clean_trans_csv, index=False)

print(f"Saved: {clean_main_csv.resolve()} ({clean_main_csv.stat().st_size:,} bytes)")
print(f"Saved: {clean_trans_csv.resolve()} ({clean_trans_csv.stat().st_size:,} bytes)")


Saved: E:\infosys\TED Talks Recommendation System\data\ted_main_cleaned.csv (7,956,118 bytes)
Saved: E:\infosys\TED Talks Recommendation System\data\transcripts_cleaned.csv (55,762,378 bytes)


## 8. Summary & Next Steps
- **Module 2 is complete!**
- The datasets have been thoroughly explored, visualized, and cleaned.
- Cleaned files `data/ted_main_cleaned.csv` and `data/transcripts_cleaned.csv` are saved and ready for **Module 3 (Feature Engineering)**.
